<a href="https://colab.research.google.com/github/eyramg/ds2002-fa26/blob/main/Copy_of_2026_09_30_%E2%80%94_JSON_to_Tidy_Tables_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

It is wrong to say the API sent a dictionary because the API sent JSON as a string, which Python converted into a dictionary after `json.loads()` parsed it.

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [5]:
print('top-level keys:', list(data.keys()))

print('portfolio name:', data['portfolio']['name'])

holdings = data['portfolio']['holdings']
print('holdings:', len(holdings))

price_rows = sum(len(h['prices']) for h in holdings)
print('price rows:', price_rows)

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio name: Cavalier Growth
holdings: 3
price rows: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: 3
- price rows: 9
- one row of the table you are about to build is one ticker's price record for one day.

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [6]:
print(data['portfolio']['holdings'][0]['company']['sector'])

try:
    print(data['portfolio']['holdings'][2]['company']['sector'])
except KeyError as e:
    print('Error:', e)

company = data['portfolio']['holdings'][2]['company']

if 'sector' in company:
    print(company['sector'])
else:
    print('unknown')

Technology
Error: 'sector'
unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [7]:
holdings = data['portfolio']['holdings']

for h in holdings:
    ticker = h['ticker']

    if 'sector' not in h['company']:
        print(ticker, '- missing sector')

    for p in h['prices']:
        if p['close'] is None:
            print(ticker, p['date'], '- close is None')

        if isinstance(p['close'], str):
            print(ticker, p['date'], '- close is a string')

        if p['volume'] == 0:
            print(ticker, p['date'], '- volume is zero')

MSFT 2026-09-29 - close is a string
MSFT 2026-09-29 - volume is zero
KO - missing sector
KO 2026-09-29 - close is None


**TODO:** your answer to the last question: _..._

In [ ]:
The genuinely missing data are Coca-Cola's `close` value of `None` and its missing sector field. Microsoft's closing price of `"498.75"` is present but stored as the wrong data type, while the volume of `0` is a real measurement and should not automatically be treated as missing.

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [9]:
holdings = data['portfolio']['holdings']

px = pd.json_normalize(
    holdings,
    record_path='prices',
    meta=[
        'ticker',
        'shares',
        ['company', 'name'],
        ['company', 'sector']
    ],
    sep='.',
    errors='ignore'
)

assert len(px) == 9

print(px)
print(px.dtypes)

         date   close    volume ticker shares     company.name company.sector
0  2026-09-28   231.4  52300000   AAPL    120       Apple Inc.     Technology
1  2026-09-29  229.85  48100000   AAPL    120       Apple Inc.     Technology
2  2026-09-30   233.1  55600000   AAPL    120       Apple Inc.     Technology
3  2026-09-28   502.1  22100000   MSFT     45  Microsoft Corp.     Technology
4  2026-09-29  498.75         0   MSFT     45  Microsoft Corp.     Technology
5  2026-09-30   505.3  24800000   MSFT     45  Microsoft Corp.     Technology
6  2026-09-28   71.25  12400000     KO    300    Coca-Cola Co.            NaN
7  2026-09-29    None  13000000     KO    300    Coca-Cola Co.            NaN
8  2026-09-30    70.9  11800000     KO    300    Coca-Cola Co.            NaN
date              object
close             object
volume             int64
ticker            object
shares            object
company.name      object
company.sector    object
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [10]:
px['close'] = pd.to_numeric(px['close'], errors='coerce')

print(px['close'].dtype)
print('null close values:', px['close'].isna().sum())

float64
null close values: 1


There is one null value remaining, and it comes from Coca-Cola's `None` closing price on 2026-09-29. The string `"498.75"` was successfully converted to a numeric value.

### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [11]:
# 1. Mean closing price and count per ticker
summary = px.groupby('ticker')['close'].agg(['mean', 'count'])
print('Mean closing price and count:')
print(summary)

# 2. Position values on 2026-09-30
last_day = px[px['date'] == '2026-09-30'].copy()

last_day['position_value'] = last_day['shares'] * last_day['close']

print('\nPosition values on 2026-09-30:')
print(last_day[['ticker', 'shares', 'close', 'position_value']])

total_value = last_day['position_value'].sum()
largest_position = last_day.loc[last_day['position_value'].idxmax(), 'ticker']

print('Total portfolio value:', total_value)
print('Largest position:', largest_position)

# 3. Mean volume with and without zero
mean_volume_all = px['volume'].mean()
mean_volume_no_zero = px.loc[px['volume'] != 0, 'volume'].mean()
difference = mean_volume_no_zero - mean_volume_all

print('\nMean volume including zero:', mean_volume_all)
print('Mean volume excluding zero:', mean_volume_no_zero)
print('Difference:', difference)

Mean closing price and count:
           mean  count
ticker                
AAPL    231.450      3
KO       71.075      2
MSFT    502.050      3

Position values on 2026-09-30:
  ticker shares  close position_value
2   AAPL    120  233.1        27972.0
5   MSFT     45  505.3        22738.5
8     KO    300   70.9        21270.0
Total portfolio value: 71980.5
Largest position: AAPL

Mean volume including zero: 26677777.777777776
Mean volume excluding zero: 30012500.0
Difference: 3334722.222222224


**TODO:** the zero-volume question: Dropping the zero-volume day was not the right call because zero is an actual reported measurement, not missing data. Excluding it would artificially increase the mean daily volume.

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [12]:
summary = px.groupby('ticker')['close'].agg(['mean', 'count']).reset_index()

result = {
    'as_of': data['as_of'],
    'currency': data['currency'],
    'portfolio': data['portfolio']['name'],
    'summary': summary.to_dict(orient='records')
}

with open('portfolio_summary.json', 'w') as f:
    json.dump(result, f, indent=2, allow_nan=False)

with open('portfolio_summary.json', 'r') as f:
    print(f.read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "portfolio": "Cavalier Growth",
  "summary": [
    {
      "ticker": "AAPL",
      "mean": 231.45000000000002,
      "count": 3
    },
    {
      "ticker": "KO",
      "mean": 71.075,
      "count": 2
    },
    {
      "ticker": "MSFT",
      "mean": 502.05,
      "count": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [13]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [14]:
# TODO A - Flatten to one row per vendor
vendors = pd.json_normalize(
    response,
    record_path=['zones', 'vendors'],
    meta=[
        'game',
        'date',
        ['zones', 'zone'],
        ['zones', 'capacity']
    ],
    sep='_',
    errors='ignore'
)

print("Vendor rows:", len(vendors))
print(vendors)


# TODO B - Find missing revenue and record whether revenue was reported
vendors['revenue_reported'] = vendors['sales_revenue'].notna()

print(vendors[['vendor_id', 'name', 'sales_revenue', 'revenue_reported']])

# Missing revenue should not be filled with zero because missing means
# the revenue was not reported, while zero would mean $0 was actually reported.


# TODO C - Flatten to one row per line item
lines = pd.json_normalize(
    response,
    record_path=['zones', 'vendors', 'lines'],
    meta=[
        ['zones', 'vendors', 'vendor_id'],
        ['zones', 'zone']
    ],
    sep='_'
)

print("Line item rows:", len(lines))
print("Quantity total:", lines['qty'].sum())
print(lines)


# TODO D - Join line items with vendor information
joined = lines.merge(
    vendors[['vendor_id', 'name', 'zones_zone', 'zones_capacity']],
    left_on=['zones_vendors_vendor_id', 'zones_zone'],
    right_on=['vendor_id', 'zones_zone'],
    how='left',
    indicator=True
)

print(joined)
print("Joined rows:", len(joined))
print(joined['_merge'].value_counts())

Vendor rows: 4
  vendor_id             name  \
0      V-01     Hoos Burgers   
1      V-07     Stadium Dogs   
2      V-12      Wahoo Wings   
3      V-18  Rally Rain Gear   

                                               lines  sales_orders  \
0  [{'item': 'Cheeseburger', 'qty': 180}, {'item'...           300   
1                  [{'item': 'Hot Dog', 'qty': 210}]           210   
2  [{'item': 'Wings 6pc', 'qty': 90}, {'item': 'W...           150   
3               [{'item': 'Rain Poncho', 'qty': 95}]            95   

   sales_revenue         game        date zones_zone zones_capacity  
0         2250.0  UVA vs Duke  2026-10-10          A            400  
1            NaN  UVA vs Duke  2026-10-10          A            400  
2         1800.0  UVA vs Duke  2026-10-10          B            150  
3          570.0  UVA vs Duke  2026-10-10          C            120  
  vendor_id             name  sales_revenue  revenue_reported
0      V-01     Hoos Burgers         2250.0              True

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [15]:
# Checkpoint
holdings_count = 3
price_rows = 9
largest_position = 'AAPL'
null_vs_zero = 'KO close was missing (None); MSFT volume of 0 was a real measurement'
zero_volume_call = 'I kept the zero in the mean because it was a real reported measurement, not missing data.'

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: KO close was missing (None); MSFT volume of 0 was a real measurement
zero volume: I kept the zero in the mean because it was a real reported measurement, not missing data.
